Date Created: 21/08/2026

Last Edited: 21/08/2026

In [0]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report

from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [0]:
import ast

In [0]:
from sklearn.calibration import CalibratedClassifierCV

# Load the datasets

In [0]:
df = pd.read_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-classification/exercises_dataset_CLEAN.csv')

In [0]:
data = pd.read_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-analysis-datasets/programs_detailed_boostcamp_kaggle.csv')

In [0]:
summary = pd.read_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-analysis-datasets/program_summary.csv')

In [0]:
data.head()

# Run Naiive Term Frequency Model

In [0]:
def norm(s):
    s = str(s).lower()
    s = re.sub(r'\(.*?\)', '', s)          # strip equipment notes in parens
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

In [0]:
labeled = df
labeled['norm'] = labeled['exercise'].apply(norm)

prog = data
unique_names = pd.DataFrame({'exercise_name': prog['exercise_name'].unique()})
unique_names['norm'] = unique_names['exercise_name'].apply(norm)

# Layer 1: exact-match bootstrap

In [0]:
lookup = labeled.drop_duplicates('norm').set_index('norm')['label']

In [0]:
unique_names['label'] = unique_names['norm'].map(lookup)
# to_predict = unique_names[unique_names['label'].isna()].copy()
# print(f"{unique_names['label'].notna().sum()} matched directly, {len(to_predict)} need prediction")

# Layer 2: Train Classifier on labeled set

In [0]:
vectorizer = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1,2), min_df=1, stop_words="english")),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=1)),
])
pipe = Pipeline([("feats", vectorizer), ("clf", LinearSVC(class_weight="balanced", max_iter=5000))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, labeled['exercise'], labeled['label'], cv=cv, scoring='f1_macro', n_jobs=-1)
print("CV macro-F1:", scores.mean(), scores.std())

pipe.fit(labeled['exercise'], labeled['label'])
to_predict['label'] = pipe.predict(to_predict['exercise_name'])

## Calibrated probabilities to estimate errors

In [0]:
calibrated = CalibratedClassifierCV(pipe, cv=5, method='sigmoid')
calibrated.fit(labeled['exercise'], labeled['label'])

probs = calibrated.predict_proba(to_predict['exercise_name'])
to_predict['confidence'] = probs.max(axis=1)
to_predict.sort_values('confidence').head(30)  # review these first

In [0]:
to_predict.shape

In [0]:
to_predict.confidence.plot.hist(bins=20)

In [0]:
to_predict.confidence.plot.kde()
plt.xlim(0,1)

In [0]:
to_predict['confidence'].plot(kind='hist', cumulative=True, density=True)

In [0]:
to_predict.to_csv('exercises_to_predict.csv')

# Labelling completed --> Allocate to original dataset

In [0]:
to_predict = pd.read_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-classification/exercises_to_predict_MANUAL_LABELED.csv')

In [0]:
to_predict.head()

In [0]:
to_predict[to_predict.exercise_name == 'Hammer Curl']

In [0]:
to_predict['final_label'] = to_predict['manual_label']
to_predict['final_label'] = to_predict['final_label'].fillna(to_predict['my_label'])

In [0]:
label_dict = dict(zip(to_predict['exercise_name'], to_predict['final_label']))

In [0]:
unique_names['model_labels'] = unique_names['exercise_name'].map(label_dict)
unique_names['label'] = unique_names['label'].fillna(unique_names['model_labels'])

In [0]:
# unique_names.to_csv('model_labels.csv', index=False)

In [0]:
program_label = dict(zip(unique_names['exercise_name'], unique_names['label']))
prog['exercise_label'] = prog['exercise_name'].map(program_label)

In [0]:
prog

In [0]:
prog.isna().sum()

# Clean up dataset (remove programs with no listed levels, equipment, goals)

TODO:
- Remove programs with no equipment
- Remove programs with no program length
- Relabel programs that have the same name (but different descriptions, levels, goals etc.)
- Recreate the program summary list
- 

In [0]:
prog.isna().sum()

In [0]:
prog[prog.equipment.isna()].title.unique()

In [0]:
prog.drop(prog[prog.title == 'Lera'].index, inplace=True)

In [0]:
prog.isna().sum()

## Find programs with multiple descriptions

In [0]:
dup_titles = prog.groupby('title')['description'].nunique().sort_values(ascending=False)
dup_titles[dup_titles > 1]

In [0]:
duplicate_titles = list(dup_titles[dup_titles > 1].index)

In [0]:
# Create a new DataFrame with the duplicate titles
prog2 = pd.DataFrame()
for title in duplicate_titles:
    desc_list = prog[(prog.title == title)].description.unique()
    tmp_title = 1
    for desc in desc_list:
        tmp = prog[(prog.title == title) & (prog.description == desc)]
        tmp['title'] = title + f' ({tmp_title})'
        prog2 = pd.concat([prog2, tmp])
        tmp_title += 1


In [0]:
dup_equips = prog2.groupby('title')['equipment'].nunique().sort_values(ascending=False)
dup_equips[dup_equips > 1]

In [0]:
prog[prog.title.isin(duplicate_titles)].shape

In [0]:
# Drop the original duplicate title rows 
prog.drop(prog[prog.title.isin(duplicate_titles)].index, inplace=True)

In [0]:
# Replace with the same rows (with new titles)
prog = pd.concat([prog, prog2])
prog.reset_index(inplace=True)

In [0]:
prog.drop(columns=['index'], inplace=True)

In [0]:
prog.to_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-analysis-datasets/programs_detailed_boostcamp_kaggle_LABELLED_CLEAN_TITLE.csv')

## Combine the Levels and Goals within the same program

I have a set mapping function, which adds in the gaps. It might be best to try to create a numerical mapping, followed by a literal translation back into the name. 

TODO: Create a straghtforward function that allows me to convert the string of levels into a set, and back into the entries of the model

The same function can be applied as goals, but in a simpler way (since goals aren't ordered)

In [0]:
for title in prog.title.unique():
    idx = prog[prog.title == title].index
    combined_level = 
    

In [0]:
title = 'Full Body Low Volume (1)'

In [0]:
prog[(prog.title == title)].level.apply(ast.literal_eval)
prog.level = prog.level.apply(set)

# Allocate list of "days" per exercise program